# Control Flow & Functions

## What's covered

The two halves: how Python decides what to do next (control flow), and how you package code into reusable units (functions). Both should already feel familiar from any other language; the goal here is to nail the Python-specific shapes that distinguish it.

**Control flow:**

- `if` / `elif` / `else` — and why there's no `switch` (until 3.10)
- The conditional expression — `x if cond else y`
- `while` loops — and the `else` clause attached to a loop
- `for` loops — and what they actually do under the hood (iteration)
- `range()` — the lazy integer sequence
- `break` and `continue`
- `for-else` and `while-else` — a feature people argue about
- `match` / `case` — structural pattern matching (Python 3.10+)
- `pass` — the no-op statement

**Functions:**

- `def` — the basic shape
- Positional vs keyword arguments
- Default parameter values — and the mutable-default trap
- `*args` and `**kwargs` — variable-length parameters
- Positional-only `/` and keyword-only `*` separators
- `return` — and what implicit `return` returns
- Docstrings and `help()`
- `lambda` — small anonymous functions
- Scopes — LEGB (Local, Enclosing, Global, Built-in)
- `global` and `nonlocal`
- Closures — functions that capture surrounding state
- First-class functions — passing them around

## Part 1 — Control Flow

## `if` / `elif` / `else`

Python uses indentation for blocks instead of braces. The colon marks the start of a block; everything indented under it is the block body. The convention is **four spaces** per indent level — never tabs, never mixed.

The shape:

```python
if condition:
    body
elif other_condition:
    body
else:
    body
```

The conditions are evaluated top-to-bottom; the first true branch wins. `elif` is short for "else if" — Python made it one keyword to keep nested if-chains flat.

In [ ]:
score = 73

if score >= 90:
    grade = "A"
elif score >= 75:
    grade = "B"
elif score >= 60:
    grade = "C"
else:
    grade = "F"

print(grade)   # C

## The conditional expression — `x if cond else y`

Python's ternary, but spelled inside-out compared to C. The whole thing is an **expression** — it produces a value, and you can use it wherever an expression is allowed.

```python
grade = "pass" if score >= 60 else "fail"
```

The order reads naturally as English: *"pass, if the score is at least 60, else fail."* Use it for small two-way choices that fit on one line. For three or more branches, the full `if`/`elif`/`else` is clearer.

In [ ]:
score = 73
label = "pass" if score >= 60 else "fail"
print(label)

# Useful in expressions where statements don't fit — e.g. list comprehensions
xs = [1, 2, 3, 4, 5]
labels = ["even" if x % 2 == 0 else "odd" for x in xs]
print(labels)

## `while` loops

`while` runs its body as long as the condition is truthy. The condition is checked *before* each iteration; if it's false at the start, the body never runs.

In [ ]:
# Classic counter
i = 0
while i < 5:
    print(i)
    i += 1

# while with a sentinel — terminate when condition flips
xs = [3, 1, 4, 1, 5, 9, 2, 6]
target = 5
i = 0
while i < len(xs) and xs[i] != target:
    i += 1
print(f"target found at index {i}" if i < len(xs) else "not found")

## `for` loops — and what they actually do

`for` in Python is not C's `for(i=0; i<n; i++)`. It iterates over a sequence — anything that produces values one at a time. Lists, tuples, dicts, sets, strings, generators, file handles, ranges — all *iterables*. The loop body runs once per element, with the loop variable bound to that element.

```python
for item in iterable:
    body
```

Under the hood, `for x in xs` calls `iter(xs)` to get an iterator, then repeatedly calls `next(iterator)` until `StopIteration` is raised. This means **you almost never need an index** to walk a sequence. If you do need indices, reach for `enumerate(...)`.

In [ ]:
# Idiomatic — just iterate the elements
words = ["alpha", "beta", "gamma"]
for w in words:
    print(w)

# When you need both index and value — enumerate
for i, w in enumerate(words):
    print(f"{i}: {w}")

# Two sequences side by side — zip
ages = [30, 25, 42]
for name, age in zip(words, ages):
    print(f"{name} -> {age}")

# Iterating a dict — by default gives keys
config = {"host": "db.local", "port": 5432, "ssl": True}
for key in config:
    print(key, config[key])

# Better — items() gives (key, value) pairs
for key, value in config.items():
    print(f"{key} = {value}")

## `range()` — the lazy integer sequence

`range(...)` produces integers without materializing a list. It's the right thing to use when you really do need a numeric loop.

| Form | Produces |
|---|---|
| `range(n)` | `0, 1, ..., n-1` |
| `range(start, stop)` | `start, start+1, ..., stop-1` |
| `range(start, stop, step)` | step-spaced sequence |

`range` is lazy and memory-cheap — `range(10**9)` allocates almost nothing. It's also indexable (`range(10)[3] == 3`) and has a length (`len(range(10)) == 10`).

In [ ]:
for i in range(5):
    print(i, end=" ")
print()

# start, stop
for i in range(10, 15):
    print(i, end=" ")
print()

# start, stop, step — including negative
for i in range(0, 20, 3):
    print(i, end=" ")
print()

for i in range(10, 0, -1):
    print(i, end=" ")
print()

# range is cheap regardless of size
huge = range(10**9)
print(len(huge), huge[0], huge[-1])

## `break` and `continue`

- **`break`** — exit the innermost enclosing loop immediately.
- **`continue`** — skip the rest of this iteration; go straight to the next one.

In [ ]:
# break — stop on first match
for x in [3, 1, 4, 1, 5, 9, 2, 6]:
    if x == 5:
        print("found 5")
        break

# continue — skip negatives, sum the rest
total = 0
for x in [3, -1, 4, -5, 2]:
    if x < 0:
        continue
    total += x
print(total)   # 9

## `for-else` and `while-else` — the loop's `else` clause

Python lets you attach an `else` block to a `for` or `while` loop. The `else` runs **only if the loop completed normally** — not if it exited via `break`.

The natural use is "search; if not found, do this":

```python
for x in xs:
    if matches(x):
        process(x)
        break
else:
    # only runs if the loop did NOT break
    handle_not_found()
```

This shape divides opinion. Some Pythonistas love it (concise search idiom); others find it confusing because `else` doesn't usually mean "if not broken." If your team isn't familiar with it, prefer a flag variable or a function with early return.

In [ ]:
# Find a value; report if missing
target = 7
xs = [3, 1, 4, 1, 5, 9, 2, 6]

for x in xs:
    if x == target:
        print(f"found {target}")
        break
else:
    print(f"{target} not in list")

## `match` / `case` — structural pattern matching (3.10+)

Python 3.10 added `match` / `case` — its take on a switch statement, with structural patterns over the matched value. The patterns can:

- Match **literals** (`case 0:`)
- Match **shapes** (`case [a, b]:` matches any 2-element sequence; `case {"key": v}:` matches a dict with that key)
- Match **types** (`case int():`)
- **Bind** values with the matched parts (`case Point(x, y):`)
- Use **`|`** for or-patterns (`case 0 | 1:`)
- Use a **guard** (`case x if x > 0:`)
- **`_`** is the wildcard (matches anything, binds nothing)

This isn't just "switch with patterns" — the case clauses can destructure deeply. It rewards reading carefully, because the cases that *look* like variable references are sometimes patterns.

In [ ]:
# Simple literal matching — like a switch
def day_name(n: int) -> str:
    match n:
        case 0: return "monday"
        case 1: return "tuesday"
        case 2: return "wednesday"
        case 3 | 4: return "later in the week"
        case _: return "weekend"

print(day_name(0))
print(day_name(3))
print(day_name(6))

# Structural matching — destructure a dict
def describe(event: dict) -> str:
    match event:
        case {"type": "click", "target": target}:
            return f"click on {target}"
        case {"type": "key", "key": k}:
            return f"key pressed: {k}"
        case {"type": "scroll", "amount": amt} if amt > 0:
            return f"scrolled down by {amt}"
        case _:
            return "unknown event"

print(describe({"type": "click", "target": "button"}))
print(describe({"type": "scroll", "amount": 50}))
print(describe({"type": "hover"}))

# Class patterns — destructure attributes
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

def quadrant(p: Point) -> str:
    match p:
        case Point(0, 0):           return "origin"
        case Point(x, 0):           return f"on x-axis at {x}"
        case Point(0, y):           return f"on y-axis at {y}"
        case Point(x, y) if x > 0 and y > 0: return "Q1"
        case Point(x, y) if x < 0 and y > 0: return "Q2"
        case Point(x, y) if x < 0 and y < 0: return "Q3"
        case Point(_, _):           return "Q4"

print(quadrant(Point(0, 0)))
print(quadrant(Point(3, 4)))
print(quadrant(Point(-1, -1)))

## `pass` — the no-op statement

`pass` does nothing. It exists because Python requires *something* in a block — an empty body is a syntax error. Use `pass` as a placeholder when you're sketching out a function, class, or branch you'll fill in later.

In [ ]:
def todo_later():
    pass    # will implement after we know the data shape

class FutureFeature:
    pass

if False:
    pass    # placeholder while you toggle conditions

print("kept going")

## Part 2 — Functions

## `def` — the basic shape

A function is declared with `def`, takes parameters in parentheses, and runs its indented body. The optional **annotations** after `:` and `->` are type hints — covered in depth in notebook 08. They are not enforced at runtime; they document, and tools like `mypy` and `pyright` check them statically.

In [ ]:
def greet(name: str) -> str:
    return f"hello, {name}"

print(greet("ganesh"))

# Multiple parameters, multiple returns
def stats(xs: list[float]) -> tuple[float, float]:
    if not xs:
        return 0.0, 0.0
    total = sum(xs)
    mean  = total / len(xs)
    return total, mean

t, m = stats([1.0, 2.0, 3.0, 4.0])
print(f"total={t}, mean={m}")

# No return statement = returns None
def shout(msg: str) -> None:
    print(msg.upper())

result = shout("hello")
print(repr(result))   # None

## Positional vs keyword arguments

Every parameter can be passed positionally (by order) or by keyword (by name). At the call site, positional arguments come first, keyword arguments after. Keyword arguments make calls with many parameters much more readable.

In [ ]:
def connect(host: str, port: int, user: str, ssl: bool) -> str:
    return f"jdbc:postgresql://{host}:{port}/?user={user}&ssl={ssl}"

# All positional — fragile when there are many
print(connect("db.local", 5432, "ganesh", True))

# All keyword — order doesn't matter, intent is clear
print(connect(user="ganesh", host="db.local", ssl=True, port=5432))

# Mixed — positional must come first
print(connect("db.local", 5432, user="ganesh", ssl=True))

## Default parameter values — and the mutable-default trap

A parameter can have a default value. Callers that omit it get the default.

The trap: **default values are evaluated *once*, at function-definition time**, not on every call. For *immutable* defaults (`0`, `None`, `""`) this doesn't matter. For **mutable defaults** (`[]`, `{}`, `set()`) it matters a lot — the same list is reused across calls, and mutations stick.

The fix: default to `None`, then construct the mutable inside the body.

In [ ]:
# WRONG — mutable default
def append_item_buggy(item, into=[]):
    into.append(item)
    return into

print(append_item_buggy(1))    # [1]
print(append_item_buggy(2))    # [1, 2] — surprise! the same list
print(append_item_buggy(3))    # [1, 2, 3]

# RIGHT — default to None, construct inside
def append_item(item, into=None):
    if into is None:
        into = []
    into.append(item)
    return into

print(append_item(1))    # [1]
print(append_item(2))    # [2] — fresh list each call
print(append_item(3))    # [3]

## `*args` and `**kwargs` — variable-length parameters

- **`*args`** — collects any extra *positional* arguments into a `tuple` named `args`.
- **`**kwargs`** — collects any extra *keyword* arguments into a `dict` named `kwargs`.

The names `args` and `kwargs` are convention — what matters is the `*` and `**`. You can rename them if it reads better in context.

At the **call site**, the same `*` and `**` are used to **unpack** an existing iterable or dict into the parameter list.

In [ ]:
# *args collects positional, **kwargs collects keyword
def show_call(label, *args, **kwargs):
    print(f"[{label}]  args={args}  kwargs={kwargs}")

show_call("a", 1, 2, 3)
show_call("b", 1, 2, foo=10, bar=20)
show_call("c", key="value")

# Unpacking — at the call site
def add(a, b, c):
    return a + b + c

nums = [1, 2, 3]
print(add(*nums))                 # 6 — unpack list into positional args

config = {"a": 10, "b": 20, "c": 30}
print(add(**config))              # 60 — unpack dict into keyword args

## Positional-only `/` and keyword-only `*`

Two markers that constrain how parameters can be passed.

- **`/`** — every parameter *before* the slash is **positional-only**. Callers cannot pass them by name.
- **`*`** — every parameter *after* the star is **keyword-only**. Callers must pass them by name.

You use these to lock down an ay-pee-eye: keyword-only for rare configuration flags (forcing callers to spell out `ssl=True` rather than passing a bare `True`), positional-only for parameters whose names are implementation details you don't want to commit to.

In [ ]:
# Mix of positional-only, normal, and keyword-only
def connect(host, port, /, user, *, ssl=False, timeout=30):
    return f"{host}:{port} user={user} ssl={ssl} timeout={timeout}"

# Allowed
print(connect("db", 5432, "ganesh"))
print(connect("db", 5432, "ganesh", ssl=True))
print(connect("db", 5432, user="ganesh", timeout=60))

# NOT allowed
# print(connect(host="db", port=5432, user="ganesh"))   # host/port are positional-only
# print(connect("db", 5432, "ganesh", True))            # ssl is keyword-only

## `lambda` — small anonymous functions

`lambda parameters: expression` produces an anonymous function value. The body is a *single expression* (no statements). Useful as a one-liner argument to higher-order functions — `key=...` on `sorted`, predicates passed to `filter`, etc.

If you find yourself writing `lambda` that's more than a short expression, define a real function with `def`. It will be more readable and easier to debug (real functions have names).

In [ ]:
# As a sort key
words = ["alpha", "zeta", "bee"]
print(sorted(words, key=lambda w: len(w)))   # by length

# As a filter — though list comprehension is usually clearer
nums = [1, 2, 3, 4, 5, 6]
print(list(filter(lambda n: n % 2 == 0, nums)))     # [2, 4, 6]
print([n for n in nums if n % 2 == 0])              # same, more idiomatic

# Lambdas stored in a dict — handy for dispatch tables
ops = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
}
print(ops["add"](3, 4))   # 7
print(ops["mul"](3, 4))   # 12

## Scopes — LEGB lookup

When you use a name in a function, Python looks for it in four scopes, in order:

1. **L**ocal — the current function's locals.
2. **E**nclosing — locals of any enclosing function, for nested defs.
3. **G**lobal — the module's top level.
4. **B**uilt-in — names like `print`, `len`, `int` from the `builtins` module.

The first scope that has the name wins. This is the **LEGB** rule.

By default, **assigning** to a name inside a function makes it *local* to that function — even if there's a same-named global. To assign to a global or enclosing-function variable, you have to declare your intent with `global` or `nonlocal`.

In [ ]:
greeting = "hello"   # global

def f():
    print(greeting)   # reads the global — fine
f()

# But this fails:
def g():
    print(greeting)   # "referenced before assignment" error here
    greeting = "hi"   # because Python sees the assignment and makes greeting local
# g()   # uncomment to see the UnboundLocalError

# Explicit opt-in to write to a global
counter = 0

def tick():
    global counter
    counter += 1

tick(); tick(); tick()
print(counter)   # 3

# nonlocal — write to an enclosing function's variable
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

c = make_counter()
print(c(), c(), c())   # 1 2 3

## Closures — capturing state in a function

The `make_counter` example above is a *closure* — a function that **captures** variables from its enclosing scope. The returned function carries a reference to the enclosing scope's `count`, even after `make_counter` has finished executing.

Closures are how Python implements "private state" without a class — useful for callbacks, decorators (notebook 07), and small stateful helpers. Each call to `make_counter` produces a fresh closure with its own captured state.

In [ ]:
# Factory that builds custom adders
def make_adder(base):
    def add(x):
        return base + x
    return add

add5  = make_adder(5)
add10 = make_adder(10)
print(add5(3))    # 8
print(add10(3))   # 13

# Each instance has its own captured `base`
print(add5.__closure__[0].cell_contents)    # 5
print(add10.__closure__[0].cell_contents)   # 10

## First-class functions

Functions are values in Python. You can:

- Assign them to variables.
- Store them in lists, dicts, sets.
- Pass them as arguments to other functions.
- Return them from other functions.

This is what makes higher-order operations (`map`, `filter`, `sorted(..., key=...)`) work, what makes decorators possible (notebook 07), and what lets you build dispatch tables of functions keyed by a string.

In [ ]:
# Function as argument
def apply(f, x):
    return f(x)

print(apply(str.upper, "hello"))          # HELLO
print(apply(lambda n: n * n, 7))          # 49

# Function as return value — already shown with make_adder

# Function in a data structure
handlers = {
    "uppercase": str.upper,
    "lowercase": str.lower,
    "reverse":   lambda s: s[::-1],
}
print(handlers["uppercase"]("ganesh"))
print(handlers["reverse"]("ganesh"))

## Docstrings and `help()`

A string literal as the **first statement** in a function (or class, or module) becomes its **docstring**, accessible at runtime via `func.__doc__` and rendered by `help(func)`. The standard library, every well-maintained third-party library, and most professional Python code base attach docstrings to public ay-pee-eyes.

The conventional shape (see PEP 257) is a one-line summary, optionally followed by a blank line and longer prose. Triple-quoted because they typically span multiple lines.

In [ ]:
def normalize_email(raw: str) -> str:
    """Strip whitespace and lowercase an email address.

    Used at sign-up to make email comparisons case-insensitive.
    Returns an empty string for an empty input.
    """
    return raw.strip().lower()

print(normalize_email("  Ganesh@Example.com "))
print(normalize_email.__doc__)
help(normalize_email)

## What's next

You now have the control-flow shapes (if, while, for, match) and the function-definition surface (defaults, args/kwargs, scopes, closures, first-class functions).

Notebook 04 is the collections deep-dive — `list`, `tuple`, `dict`, `set`, the operations that work on each, and the comprehension syntax that turns most explicit loops into one-liners. It's where Python's "expressive in fifty characters" reputation gets earned.